# Crescent Bakery: Correlation and A/B Testing

Lesson 1.7 Practice. Computes correlations on the enrollment dataset,
demonstrates a confounded correlation in synthetic data, simulates an A/B
test, and walks through power calculations.

Author: [A. Chris Yi]
Date: [2026-09-24]

# Correlation Analysis

Compute Pearson and Spearman correlations between hours_studied and the numeric value of final_grade (you'll need to map the letter grades to numbers: F=0, D=1, C=2, B=3, A=4 is one reasonable encoding).

Report:

Both correlation coefficients with p-values
A scatter plot of the two variables with a trend line
A short Markdown interpretation: do the correlations agree? Is the relationship plausibly causal (does studying more hours cause a higher grade)? If you wanted to claim causation, what would you need to do?

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px

In [19]:
enrollments = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")


In [20]:
def final_grade_numeric(grade):
    grade_map = {
        "A": 4,
        "B": 3,
        "C": 2,
        "D": 1,
        "F": 0
    }
    return grade_map.get(grade, None)

final_grade_numeric = enrollments["final_grade"].apply(final_grade_numeric)

In [21]:
pearson_r, pearson_p = stats.pearsonr(
    enrollments["hours_studied"],
    final_grade_numeric,
)

spearman_r, spearman_p = stats.spearmanr(
    enrollments["hours_studied"],
    final_grade_numeric,
)

print(f"Pearson r:  {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Spearman r: {spearman_r:.3f} (p = {spearman_p:.4f})")

Pearson r:  -0.077 (p = 0.4435)
Spearman r: -0.149 (p = 0.1397)


In [22]:
fig = px.scatter(
    enrollments,
    x="hours_studied",
    y=final_grade_numeric,
    trendline="ols",
    title="Correlation between Hours Studied and Final Grade",
    labels={"hours_studied": "Hours Studied", "y": "Final Grade (Numeric)"},
)
fig.show()

# Interpretation
From the calculations and the plot, it can be determined that there is no detectable correlation between the hours studied and the final grade achieved by the student. 

# Confounder Hypothesis

In one Markdown cell, propose at least two plausible confounders that could create a correlation between hours_studied and final_grade even if studying didn't cause learning. Describe each in one or two sentences.

Students who spend more time studying could either be innateley better students dedicating more time to their studies than others or they could be students who struggle to retain/learn material and needed more time to absorb the content. 

If a student is just a naturally stronger student with high motivation and good study skills, they may dedicate more time to studying than an average student would. This would account for more time spent getting better scores.

If a student naturally struggles and needs more time to learn material, the hours studied might not impact the grades as much as in the previous situation due to struggles with learning or retention. 

# A/B Test Design

Skyline is considering adding a feature: a built-in study planner that automatically schedules study sessions for enrolled students. They want to know whether the planner causes higher final grades.

Design an A/B test to answer this. In Markdown, specify:

The treatment and control conditions
The randomization unit (per-student, per-enrollment, per-cohort?)
The outcome metric
The minimum effect size that would be practically meaningful (your judgment; defend it briefly)
The required sample size, computed from a power analysis at α=0.05 and 80% power
For the sample size calculation, you can assume the baseline final grade rate of "B or higher" is 40% in your generated data (compute the actual baseline if you want to be precise) and that you want to detect a lift to 45%.

In [26]:
np.random.seed(43)
n = 500
engagement = np.random.normal(loc=0, scale=1, size=n)
treatment = np.random.choice([0, 1], size=n)
true_treatment_effect = -0.2  # in z-score units
conversion_propensity = engagement + true_treatment_effect * treatment + np.random.normal(loc=0, scale=0.5, size=n)
converted = (conversion_propensity > 0).astype(int)

ab_test = pd.DataFrame({
    "treatment": treatment,
    "converted": converted,
})

ab_results = ab_test.groupby("treatment")["converted"].agg(["mean", "count"])
ab_results.index = ["Control", "Treatment"]
print("A/B test conversion rates:")
print(ab_results)

control_conv = ab_test[ab_test["treatment"] == 0]["converted"]
treatment_conv = ab_test[ab_test["treatment"] == 1]["converted"]

n_control = len(control_conv)
n_treatment = len(treatment_conv)
p_control = control_conv.mean()
p_treatment = treatment_conv.mean()

p_pool = (control_conv.sum() + treatment_conv.sum()) / (n_control + n_treatment)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_control + 1 / n_treatment))
z = (p_treatment - p_control) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

se_diff = np.sqrt(p_control * (1 - p_control) / n_control + p_treatment * (1 - p_treatment) / n_treatment)

diff = p_treatment - p_control
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print(f"\nTreatment effect: {diff:.3f}")

print(f"95% confidence interval: ({ci_lower:.3f}, {ci_upper:.3f})")
print(f"z = {z:.3f}, p = {p_value:.4f} ")



A/B test conversion rates:
               mean  count
Control    0.547718    241
Treatment  0.389961    259

Treatment effect: -0.158
95% confidence interval: (-0.244, -0.071)
z = -3.533, p = 0.0004 


# Treatment and Control
Treatment group has the built in planner enabled
Control group has current experience with no planner

# Randomization
Per student randomization keeps the experience consistent and doesnt introduce the possible changes in study habits from the planner on one course as an additional variable

# Outcome Metric
The primary metric is binary (final_grade B or higher) but a secondary metric needs to be retained for those who don't score a B or higher with or without the planner. 

# Minimum Effect Size
A minimum effect of +5 (ideally from 40% - 45%) is ideal. With grades coming back as a noisy dataset, anything smaller would be hard to catch in analysis and anything bigger might miss something as study tools are traditionally modestly effective. In this example it would be best to use +5 because 1 extra B per 20 students would raise it that much.